In [ ]:
# =========================================================
# 1. Imports & Dataset Loading
# =========================================================

import re

import numpy as np
import pandas as pd
from IPython.display import display


# Load the dataset
file_path = "/home/ubuntu/bolttech-prac/data/original_data/claim_use_case_dataset_vF.xlsx"
df = pd.read_excel(file_path)


# =========================================================
# 2. Component Columns Preprocessing
# =========================================================

#component_cols = [
#    "turnOnOff",
#    "touchScreen",
#    "smashed",
#    "frontCamera",
#    "backCamera",
#    "frontOrBackCamera",
#    "audio",
#    "mic",
#    "buttons",
#]
#df[component_cols] = df[component_cols].astype("boolean")

#is_phone_call = df["channel"].astype("string").str.strip() == "Phone Call"
#is_theft = df["claimType"].astype("string").str.strip() == "Theft"
#condition = is_phone_call | is_theft

#df[component_cols] = df[component_cols].astype("object")
#df.loc[condition, component_cols] = "NA"

# =========================================================
# 3. Device Type Configuration
# =========================================================

# Define required columns for device type preprocessing
REQUIRED_COLS = ["productName", "deviceType"]

# Define aliases used to standardize device types
DEVICE_ALIASES = {
    "SMARTPHONES": [
        "SMARTPHONE", "SMARTPHONES", "SMART_PHONE", "SMART_PHONES",
        "MOBILE_PHONE", "MOBILE_PHONES", "MOBILE", "PHONE", "PHONES",
        "HANDSET", "HANDSETS"
    ],
    "TABLETS": [
        "TABLET", "TABLETS", "PAD", "PADS"
    ],
    "WEARABLES": [
        "WEARABLE", "WEARABLES", "WATCH", "WATCHES", "SMARTWATCH",
        "SMARTWATCHES", "SMART_WATCH", "SMART_WATCHES", "WATCH6", "WATCH_6"
    ],
    "LAPTOPS": [
        "LAPTOP", "LAPTOPS", "NOTEBOOK", "NOTEBOOKS"
    ],
    "DESKTOPS": [
        "DESKTOP", "DESKTOPS", "PC", "PCS"
    ],
    "EARBUDS": [
        "EARBUD", "EARBUDS", "BUD", "BUDS", "EARPHONE", "EARPHONES",
        "HEADPHONE", "HEADPHONES", "HEADSET", "HEADSETS"
    ],
}


# =========================================================
# 4. Device Type Helper Functions & Pre-computation
# =========================================================

def normalize_text(x):
    """Standardize text by converting it to uppercase and replacing non-alphanumeric characters with underscores."""
    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()
    x = re.sub(r"[^A-Z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x)

    return x.strip("_")


# Pre-normalize aliases and pre-compile regex patterns to avoid repeated computation
NORMALIZED_ALIASES = {}
COMPILED_PATTERNS = {}

for standard_device, aliases in DEVICE_ALIASES.items():
    # Pre-normalize all aliases for strict matching
    norm_aliases = [normalize_text(alias) for alias in aliases if pd.notna(alias)]
    NORMALIZED_ALIASES[standard_device] = norm_aliases

    # Combine all aliases into a single compiled regex pattern for fast token searching
    escaped_aliases = "|".join(re.escape(alias) for alias in norm_aliases)
    COMPILED_PATTERNS[standard_device] = re.compile(rf"(^|_)(?:{escaped_aliases})($|_)")


def extract_device_from_name(product_name):
    """Extract the standardized device type from productName using pre-compiled regex patterns."""
    if pd.isna(product_name):
        return pd.NA

    text = normalize_text(product_name)

    for standard_device, pattern in COMPILED_PATTERNS.items():
        if pattern.search(text):
            return standard_device

    return pd.NA


def normalize_device_type(device_type):
    """Map an existing deviceType value to the standardized device type categories."""
    if pd.isna(device_type):
        return pd.NA

    text = normalize_text(device_type)

    for standard_device, norm_aliases in NORMALIZED_ALIASES.items():
        if text == standard_device or text in norm_aliases:
            return standard_device

    return text


# =========================================================
# 5. Device Type Transformation Pipeline
# =========================================================

# Validate required columns
missing_cols = [col for col in REQUIRED_COLS if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Replace empty or whitespace-only strings with NaN
df[REQUIRED_COLS] = df[REQUIRED_COLS].replace(r"^\s*$", np.nan, regex=True)

# Backup original deviceType and derive standardized device type information
df["deviceType_original"] = df["deviceType"]
df["deviceType_from_productName"] = df["productName"].apply(extract_device_from_name)
df["deviceType_normalized"] = df["deviceType_original"].apply(normalize_device_type)

# Fill missing original deviceType using device type extracted from productName
df["deviceType_filled"] = df["deviceType_original"].copy()

mask_fill = (
    df["deviceType_filled"].isna()
    & df["deviceType_from_productName"].notna()
)

df.loc[mask_fill, "deviceType_filled"] = df.loc[
    mask_fill,
    "deviceType_from_productName"
]

df["deviceType"] = df["deviceType_filled"]

# Check consistency between normalized original deviceType and productName-derived device type
mask_comparable = (
    df["deviceType_normalized"].notna()
    & df["deviceType_from_productName"].notna()
)

df["deviceType_matches_productName"] = (
    df["deviceType_normalized"]
    .eq(df["deviceType_from_productName"])
    .where(mask_comparable, pd.NA)
    .astype("boolean")
)

# Create mismatch cases for manual inspection
device_type_mismatch = df.loc[
    df["deviceType_matches_productName"] == False,
    [
        "productName",
        "deviceType_original",
        "deviceType_normalized",
        "deviceType_from_productName",
        "deviceType_matches_productName",
    ],
].copy()

# Create productName extraction failure cases for manual inspection
product_name_unmatched = df.loc[
    df["productName"].notna() & df["deviceType_from_productName"].isna(),
    [
        "productName",
        "deviceType_original",
        "deviceType_from_productName",
    ],
].copy()

# Create suggested deviceType based on normalized deviceType and productName-derived device type
df["deviceType_suggested"] = df["deviceType_normalized"]

mask_use_product_name = df["deviceType_from_productName"].notna()

df.loc[mask_use_product_name, "deviceType_suggested"] = df.loc[
    mask_use_product_name,
    "deviceType_from_productName"
]


# =========================================================
# 6. Device Type Reporting & Diagnostics
# =========================================================

# Print execution summary
print("\n" + "=" * 40)
print(" Execution Summary Report")
print("=" * 40)
print(f"Total rows processed: {len(df)}")
print(f"Missing deviceType (Original): {df['deviceType_original'].isna().sum()}")
print(f"Successfully extracted from productName: {df['deviceType_from_productName'].notna().sum()}")
print(f"Missing deviceType filled by productName: {mask_fill.sum()}")
print(f"Comparable cases (Both present): {mask_comparable.sum()}")
print(f"Matching cases: {(df['deviceType_matches_productName'] == True).sum()}")
print(f"Mismatched cases: {(df['deviceType_matches_productName'] == False).sum()}")
print(f"Extraction failures from productName: {len(product_name_unmatched)}")
print(f"Missing deviceType (Final): {df['deviceType'].isna().sum()}")

# Print value distribution reports
dist_headers = [
    ("Original deviceType Distribution", "deviceType_original"),
    ("Extracted deviceType Distribution", "deviceType_from_productName"),
    ("Final deviceType Distribution", "deviceType"),
    ("Suggested deviceType Distribution", "deviceType_suggested"),
]

for title, col in dist_headers:
    print("\n" + "=" * 40)
    print(title)
    print("=" * 40)
    print(df[col].value_counts(dropna=False))

# Display mismatch cases for manual inspection
print("\n" + "=" * 40)
print(f"Mismatched Cases (Total: {len(device_type_mismatch)})")
print("=" * 40)
display(device_type_mismatch.head(100))

# Display productName extraction failure cases for manual inspection
print("\n" + "=" * 40)
print(f"Extraction Failures (Total: {len(product_name_unmatched)})")
print("=" * 40)
display(product_name_unmatched.head(100))


# =========================================================
# 7. Final Device Type Override
# =========================================================

# Keep the same final overwrite behavior as the original code
df["deviceType"] = df["deviceType_from_productName"]


# =========================================================
# 8. Date Feature Engineering
# =========================================================

# Define date columns
date_cols = [
    "policyStartDate",
    "policyEndDate",
    "purchaseDate",
]

# Convert date columns to datetime
for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        dayfirst=True,
        errors="coerce",
    ).dt.normalize()

# Calculate policy duration in days
df["policy_duration_days"] = (
    df["policyEndDate"] - df["policyStartDate"]
).dt.days

# Calculate raw policy duration in months
df["policy_duration_months_raw"] = (
    (df["policyEndDate"].dt.year - df["policyStartDate"].dt.year) * 12
    + (df["policyEndDate"].dt.month - df["policyStartDate"].dt.month)
)

# Subtract one month when the end day is earlier than the start day
df.loc[
    df["policyEndDate"].dt.day < df["policyStartDate"].dt.day,
    "policy_duration_months_raw",
] -= 1

df["days_from_purchase_to_policy"] = df["policyStartDate"] - df["purchaseDate"]
df["days_from_purchase_to_policy"] = df["days_from_purchase_to_policy"].dt.days


def map_policy_months(x):
    """Map raw policy duration to the closest valid policy duration category."""
    if pd.isna(x):
        return np.nan

    valid_months = np.array([6, 12, 24])

    return valid_months[np.argmin(np.abs(valid_months - x))]


# Map raw policy duration to 6, 12, or 24 months
df["policy_duration_months"] = df["policy_duration_months_raw"].apply(map_policy_months)

# Calculate the gap between purchase date and policy start date
df["purchase_to_policy_start_days"] = (
    df["policyStartDate"] - df["purchaseDate"]
).dt.days

# Calculate the gap between purchase date and policy end date
df["purchase_to_policy_end_days"] = (
    df["policyEndDate"] - df["purchaseDate"]
).dt.days

# Create policy start cohort features
df["policy_start_year"] = df["policyStartDate"].dt.year
df["policy_start_month"] = df["policyStartDate"].dt.month

# Create purchase cohort features
df["purchase_year"] = df["purchaseDate"].dt.year
df["purchase_month"] = df["purchaseDate"].dt.month


# =========================================================
# 9. RRP-Based Feature Engineering
# =========================================================

# Create RRP-based features
df["remaining_rrp_ratio"] = df["oldBalanceRRP"] / df["rrp"]
df["used_rrp_amount"] = df["rrp"] - df["oldBalanceRRP"]
df["has_prior_rrp_usage"] = df["used_rrp_amount"] > 0


# =========================================================
# 10. Final Feature Selection
# =========================================================

# Define final columns to keep
valid_cols = [
    "excessFee",
    "rrp",
    "oldBalanceRRP",
    "remaining_rrp_ratio",
    "used_rrp_amount",
    "has_prior_rrp_usage",
    "coverage",
    "policy_duration_months",
    "policy_start_year",
    "policy_start_month",
    "days_from_purchase_to_policy",
    "retailerName",
    "deviceType",
    "channel",
    "claimType",
    "country",
    "status",
    "turnOnOff",
    "touchScreen",
    "smashed",
    "frontCamera",
    "backCamera",
    "frontOrBackCamera",
    "audio",
    "mic",
    "buttons",
    "other",
    "issueDesc",
]

# Create the final feature dataset
valid_df = df[valid_cols]


# =========================================================
# 11. Export Feature Dataset
# =========================================================

# Export the final feature dataset to Excel
output_path = "claim_approval_feature_dataset.xlsx"
valid_df.to_excel(output_path, index=False)